In [ ]:
import ray
ray.shutdown()

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import pandas as pd
from datasets import load_from_disk
import altair as alt
from collections import defaultdict, Counter
import random 
import os
import csv
import numpy as np
alt.data_transformers.enable("vegafusion")
from fuzzywuzzy import fuzz, process
import ray
ray.init(ignore_reinit_error=True)

In [ ]:
llm_extract = load_from_disk("../pmc_llmExtraction_v8") 
extract = llm_extract.to_list()
def parse_nested_json(value):
    while isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            break
    return value

for item in extract:
    if isinstance(item["extraction"], str):
        item["extraction"] = parse_nested_json(item["extraction"])
    if isinstance(item["case"], str):
        item["case"] = parse_nested_json(item["case"])

In [ ]:
def combined_corpus(dic):
    if isinstance(dic["case"], dict):
        dic["case"] = " ".join(dic["case"].values())
    return dic

# Apply the function to each item in extract
preprocessed_extract = [combined_corpus(i) for i in extract]
# Filter out no extraction
preprocessed_extract = [item for item in preprocessed_extract if item["extraction"]]

In [ ]:
def parse_nested_json(value):
    """Recursively parses a JSON string until it is a proper dictionary/list."""
    while isinstance(value, str):  # Ensure value is not a JSON string
        try:
            value = json.loads(value)  # Convert JSON string to dict or list
        except json.JSONDecodeError:
            break  # If it's not JSON, return the original value
    return value

# Ensure extract is a list of dictionaries
if isinstance(extract, str):  
    extract = json.loads(extract)  # Convert JSON string to list

elif isinstance(extract, list) and all(isinstance(item, str) for item in extract):
    extract = [json.loads(item) for item in extract]  # Convert each string to a dict

# Process each dictionary in the list
for item in extract:
    if isinstance(item, dict):  # Ensure item is a dictionary
        if "extraction" in item and isinstance(item["extraction"], str):
            item["extraction"] = parse_nested_json(item["extraction"])

        if "case" in item and isinstance(item["case"], str):
            item["case"] = parse_nested_json(item["case"])
 

In [ ]:
# Ensure no case contains empty string ""
preprocessed_extract = [item for item in preprocessed_extract if item["case"] != [""]]

# Word Counts

In [ ]:
with open("../data/merged_extracted_sex.json") as f:
    sex = json.load(f)
    
sex_df = pd.DataFrame(list(sex.items()), columns=["pmcid", "sex"])

sex_df["Sex"] = sex_df["sex"].replace({None: "unspecified"}) # convert failed to extraction to "unspecified" 
sex_df["pmcid"] = sex_df["pmcid"].astype(str)

male_pmcid = set(sex_df[sex_df["Sex"]=="male"]["pmcid"].tolist())

# Convert the male PMCID list to a set for fast lookups (O(1) instead of O(n))
male_pmcid_set = set(male_pmcid)

# Efficiently update 'Pregnancy' in place
for item in preprocessed_extract:
    if item["pmcid"] in male_pmcid_set:
        item["extraction"].setdefault("Pregnancy", [])[:] = []  # Ensures empty list

In [ ]:
categories = ['Vitals_Hema', 'Pregnancy', 'Neuro', 'CVS', 'RESP', 'EENT', 
              'GI', 'GU', 'DERM', 'MSK', 'ENDO', 'LYMPH', 'History', 'Lab_Image']

 
count_df = pd.DataFrame([
    {
        "pmcid": entry["pmcid"],
        "title": entry["title"],
        "total_categories": len(entry["extraction"]),  # Count number of categories in extraction
        "total_extraction_items": sum(len(v) for v in entry["extraction"].values()),  # Total extraction items
        "case_length": len(entry["case"].split()),  # Word count of case text
        **{
            category: sum(len(item.split()) for item in entry["extraction"].get(category, [])) 
            for category in categories
        }  # Word count for each category
    }
    for entry in preprocessed_extract
])

 

In [ ]:
count_df.describe().apply(lambda x: round(x,2))

In [ ]:
filtered_df = count_df.drop(columns=["title", "total_categories", "total_extraction_items"])
filtered_df.rename(columns={"case_length":"Case Length"}, inplace=True)
# Convert the DataFrame into a long format for Altair
count_df_long = filtered_df.melt(id_vars=["pmcid"], 
                              var_name="Category", 
                              value_name="Word_Count")
count_df_long = count_df_long[count_df_long["Word_Count"] > 0]

category_order = ["Case Length"] + [col for col in filtered_df.columns if col not in ["pmcid", "Case Length"]]
box_plot = alt.Chart(count_df_long).mark_boxplot(
    color="steelblue",    # Nicer color
    size=20               # Adjust the width of the box plot
).encode(
    x=alt.X("Category:N", title="Clinical Categories", sort=category_order, axis=alt.Axis(labelAngle=-45)),  # Rotate x-axis labels
    y=alt.Y("Word_Count:Q", title="Word Count per Extraction", scale=alt.Scale(type="log")),  # Log scale for readability
    tooltip=["Category", "Word_Count"]
).properties(
    # title="Enhanced Distribution of Word Counts per Clinical Category",
    width=900,  # Wider chart for better spacing
    height=500
).configure_axis(
    grid=True,  # Enable grid for readability
    labelFontSize=12,
    titleFontSize=14
).configure_title(
)

box_plot.save("../graphs/word_count_boxplot.pdf")
box_plot

# Exact Match

In [ ]:
def parsing(item):
    parts = item.split(":")
    item = parts[-1]
    return item

In [ ]:
def exact_match(dic):
    match = defaultdict(list)
    case_text = dic["case"].lower() if isinstance(dic.get("case"), str) else ""
    for k, values in dic.get("extraction", {}).items():
        if k != "iem":
            # Only process items that are strings and non-empty
            match[k] = [parsing(string.lower()) in case_text for string in values if isinstance(string, str) and string != ""]
    
    return match

In [ ]:
# To perform exact match, we want to see if the value of the key-value pair of each value is indeed from the original text
e_match = [exact_match(item) for item in preprocessed_extract]

In [ ]:
e_match

In [ ]:
category_counts = defaultdict(Counter)

for dic in e_match:
    for category, values in dict(dic).items():
        category_counts[category].update(values)
category_counts = {k: dict(v) for k, v in category_counts.items()}

In [ ]:
category_counts 

In [ ]:
accuracy_dict = {k: round(v[True]/(v[True] + v[False]), 2) for k, v in category_counts.items() if k} 

In [ ]:
sum_dict = {k: v[False]+v[True] for k, v in category_counts.items() if k }

In [ ]:
category_count_df = pd.DataFrame(sum_dict.items(), columns=["category", "counts"])
chart = alt.Chart(category_count_df).mark_bar().encode(
    x=alt.X("category:N", sort='-y', title='Category'),
    y=alt.Y("counts:Q", title='Total Extracted String Counts'),
    color=alt.Color("category:N", legend=None),
    tooltip=["category", "counts"]
).properties(
    width=400,
    height=400,
).configure_axis(
    labelAngle=45
)
chart.save("../graphs/Dataset_EDA_Extractions_PerCategory.pdf")
chart

In [ ]:
category_count_df.describe()

In [ ]:
accuracy_df = pd.DataFrame(list(accuracy_dict.items()), columns=["Category", "Accuracy"])

In [ ]:
accuracy_df.describe()

In [ ]:
# Percentage Exact Matches for Each Category
em_chart = alt.Chart(accuracy_df).mark_bar().encode(
    x=alt.X("Category:N", sort='-y', title='Category'),
    y=alt.Y("Accuracy:Q", title='Exact Match'),
    color=alt.Color("Category:N", legend=None),
    tooltip=["Category", "Accuracy"]
).properties(
    width=400,
    height=400
).configure_axis(
    labelAngle=45,
    labelFontSize=20,  # Increase axis label size
    titleFontSize=18
)
em_chart.save("../graphs/exact match percentage per category.pdf")
em_chart


## FuzzyWuzzy Token Set Ratio -Per Category

In [ ]:
# for the fuzzy wuzzy (Token Set Ratio)match, like in exact match, we only check the string after the : (we make the assumption that the keys are either not relevant or always correct)
@ray.remote
def imperfect_match(dic):
    fuzzy_match = defaultdict(list)
    case_text = dic["case"].lower() if isinstance(dic.get("case"), str) else ""
    for k, values in dic.get("extraction", {}).items():
        if k != "iem":
            fuzzy_match[k] = [fuzz.token_set_ratio(parsing(string.lower()), case_text) for string in values if isinstance(string, str) and string!=""]
    return {"fuzzy_match_score": dict(fuzzy_match)}     

In [ ]:
futures = [imperfect_match.remote(dic) for dic in preprocessed_extract]

results = ray.get(futures)

In [ ]:
for dic, r in zip(preprocessed_extract, results):
    dic.update(r)

In [ ]:
fuzzy_score_by_category = defaultdict(list)
for item in results:
    for category, scores in item["fuzzy_match_score"].items():
        fuzzy_score_by_category[category].extend(scores)

In [ ]:
fuzzy_score_by_category = dict(fuzzy_score_by_category)
fuzzy_df = pd.DataFrame([(category, score) for category, scores in fuzzy_score_by_category .items() for score in scores], columns=['Category', 'Score'])

In [ ]:
categories = list(fuzzy_score_by_category.keys())
means = np.array([np.mean(scores) for scores in fuzzy_score_by_category.values()])
std_devs = np.array([np.std(scores) for scores in fuzzy_score_by_category.values()])


upper = np.clip(means + std_devs, None, 100)  # Cap the upper limit at 100
lower = np.clip(means - std_devs, 0, None) 

stats_df = pd.DataFrame({
    'Category': categories,
    'Mean': means,
    'StdDev': std_devs,
    'Upper': upper,
    'Lower': lower
})

sorted_stats_df = stats_df.sort_values(by="Mean", ascending=False)
ordered_categories = sorted_stats_df['Category'].tolist()

In [ ]:
bar_plot = alt.Chart(sorted_stats_df).mark_bar().encode(
    x=alt.X('Category:N', title="Category", sort=ordered_categories),
    y=alt.Y('Mean:Q', title="Mean TSR(%)"),
    color=alt.Color('Category:N', legend=None),
    tooltip=["Category", "Mean"]
).properties(
    width=400,
    height=400,
    # title="Mean Fuzzy Score per Category"
).configure_axis(
    labelAngle=45,
    labelFontSize=20,  # Increase axis label size
    titleFontSize=18
)

# error_bars = alt.Chart(sorted_stats_df).mark_errorbar().encode(
#     x=alt.X('Category:N', sort=ordered_categories),
#     y=alt.Y('Lower:Q', title="Mean TSR(%)"),
#     y2='Upper:Q'
# )

# Combine the bar plot with custom error bars
# bar_plot_with_error = bar_plot + error_bars
# bar_plot_with_error.save('../graphs/tokensetratio.pdf')
# bar_plot_with_error


bar_plot.save('../graphs/tokensetratio.pdf')
bar_plot

In [ ]:
stats_df["Mean"].describe()

In [ ]:
stats_df["Mean"]

In [ ]:
# Select the Highest Quality LLM Outputs based on the highest average scores from each category
def calculate_mean_fuzzy_score(extraction):
    scores = [score for cat, scores in extraction.items() for score in scores]
    return np.mean(scores)
    

In [ ]:
avg_fuzzy_score_results = [{k: round(calculate_mean_fuzzy_score(v), 3)} for dic in results for k, v in dic.items()]

for dic, r in zip(preprocessed_extract, avg_fuzzy_score_results):
    dic.update(r)
    
df_fuzzy_score_per_case = pd.DataFrame(list(dict(Counter([dic['fuzzy_match_score'] for dic in preprocessed_extract])).items()), columns=["fuzzy_avg_score", "counts"])

In [ ]:
chart = alt.Chart(df_fuzzy_score_per_case).mark_bar().encode(
    x=alt.X("fuzzy_avg_score:N", bin=alt.Bin(step=10), title="fuzzy average score"),
    y=alt.Y("counts:Q", title="counts"),
).properties(
    width=600,
    height=400,
    title="Fuzzy Score Distribution"
)

chart
    

In [ ]:
df_fuzzy_score_per_case.describe()